# The Arrow and the Coordinate

### What separates spending a resource from changing the class — and why every "currency" in this programme was the second thing wearing the first thing's clothes

**Dean Kulik — QuHarmonics Research Group**
CC BY-NC 4.0

---

## The question that was left standing

The last notebook ended on the honest open problem: adding delay *felt* like spending a resource and *behaved*
like changing the class. Postselection had the same ambiguity — it dissolved exactness, but it also changed
what counts as an operation. Until that distinction is settled, "price" and "ceiling" are labels applied by
taste.

The distinction has a computable form:

$$\boxed{\;\text{Let }\mathcal T_\Sigma\text{ be the set of transformations a specification admits.}\;}$$
$$\text{Resource extension: } \mathcal T_{E(\Sigma)} = \mathcal T_\Sigma \quad\text{— same set, different point reached in it.}$$
$$\text{Class extension: } \mathcal T_{E(\Sigma)} \supsetneq \mathcal T_\Sigma \quad\text{— strictly more transformations exist.}$$

That is not a definition to argue over; it is something to measure, because the achievable set has an
observable. For the cloning problem the observable is the **achievable $(F_1,F_2)$ frontier**: if the set grows,
the frontier moves out; if the set is the same, the frontier is identical.

And the criterion should do more than classify after the fact. If ancilla dimension is a resource, it must
**saturate where the set completes** — and Stinespring says exactly where that is: the minimal environment
dimension realizing a channel is its **Choi rank**. So the criterion makes a prediction about a number it was
not told.

Three things to establish:

1. Does the frontier stop moving, and where?
2. Does the Choi rank of the optimal channel predict that point?
3. Does postselection instead move the operation *out of the set*, demonstrably?

---

## Step 1 — Library


In [1]:
import numpy as np
from scipy.optimize import minimize, brentq

SIX = [np.array([1, 0]), np.array([0, 1]),
       np.array([1, 1]) / np.sqrt(2), np.array([1, -1]) / np.sqrt(2),
       np.array([1, 1j]) / np.sqrt(2), np.array([1, -1j]) / np.sqrt(2)]

def U_of(p, d):
    n = d * d
    Hm = p[:n].reshape(d, d) + 1j * p[n:2*n].reshape(d, d)
    Hm = Hm + Hm.conj().T
    ev, V = np.linalg.eigh(Hm)
    return V @ np.diag(np.exp(1j * ev)) @ V.conj().T

def fids(U, da):
    """Haar-averaged fidelity of each output marginal against the input."""
    F1 = F2 = 0.0
    for psi in SIX:
        v = np.kron(np.kron(psi, np.array([1, 0])), np.eye(da)[0])
        T = (U @ v).reshape(2, 2, da)
        r1 = np.einsum('ijk,ljk->il', T, T.conj())
        r2 = np.einsum('jik,jlk->il', T, T.conj())
        F1 += np.real(psi.conj() @ r1 @ psi)
        F2 += np.real(psi.conj() @ r2 @ psi)
    return F1 / 6, F2 / 6

def optimize(obj, da, restarts=4, seed=0, maxiter=600):
    """Maximize an objective of (F1,F2) over unitaries on 2 x 2 x da. Returns (fids, U)."""
    d = 4 * da; rng = np.random.default_rng(seed); bp = None; bv = -9.0
    for _ in range(restarts):
        r = minimize(lambda p: -obj(*fids(U_of(p, d), da)),
                     rng.normal(0, 0.6, 2 * d * d), method='L-BFGS-B',
                     options={'maxiter': maxiter, 'ftol': 1e-13})
        f = fids(U_of(r.x, d), da); v = obj(*f)
        if v > bv: bv, bp = v, r.x
    return fids(U_of(bp, d), da), U_of(bp, d)


# ===========================================================================
# Choi matrix of the induced channel  rho -> Tr_anc[ U (rho (x) |00><00|) U^dag ]
# for a map 1 qubit -> 2 qubits.  dim 2*4 = 8.
# Stinespring: minimal environment dimension realizing a channel = rank(J).
# ===========================================================================
def choi_from_unitary(U, da):
    J = np.zeros((8, 8), complex); E = np.eye(2)
    for i in range(2):
        for j in range(2):
            vi = np.kron(np.kron(E[i], np.array([1, 0])), np.eye(da)[0])
            vj = np.kron(np.kron(E[j], np.array([1, 0])), np.eye(da)[0])
            Ti = (U @ vi).reshape(4, da); Tj = (U @ vj).reshape(4, da)
            J[i*4:(i+1)*4, j*4:(j+1)*4] = Ti @ Tj.conj().T      # trace out the ancilla
    return J

def choi_from_kraus(A):
    J = np.zeros((8, 8), complex); E = np.eye(2)
    for i in range(2):
        for j in range(2):
            J[i*4:(i+1)*4, j*4:(j+1)*4] = np.outer(A @ E[i], (A @ E[j]).conj())
    return J

def trace_out_output(J):
    """Tr_out J. Equals I exactly for trace-preserving maps; <= I for trace non-increasing."""
    T = np.zeros((2, 2), complex)
    for i in range(2):
        for j in range(2):
            T[i, j] = np.trace(J[i*4:(i+1)*4, j*4:(j+1)*4])
    return T


# ===========================================================================
# Exact probabilistic cloner, built as ONE Kraus operator from the dual basis:
#   A = sqrt(p) ( |psi1 psi1><dual1| + |psi2 psi2><dual2| ),  <dual_i|psi_j> = delta_ij
# so A|psi_i> = sqrt(p) |psi_i psi_i> exactly. Admissible iff A^dag A <= I.
# ===========================================================================
def cloner_kraus(x, p):
    p1 = np.array([1.0, 0.0]); p2 = np.array([x, np.sqrt(1 - x * x)])
    P = np.column_stack([p1, p2])
    Dl = P @ np.linalg.inv(P.conj().T @ P)              # columns: the dual vectors
    A = np.sqrt(p) * (np.outer(np.kron(p1, p1), Dl[:, 0].conj())
                      + np.outer(np.kron(p2, p2), Dl[:, 1].conj()))
    return A, p1, p2

def max_p_by_construction(x):
    """Largest p with A^dag A <= I. Found from the construction, not from a formula."""
    f = lambda p: np.linalg.eigvalsh(cloner_kraus(x, p)[0].conj().T
                                     @ cloner_kraus(x, p)[0]).max() - 1.0
    return brentq(f, 1e-9, 1.0, xtol=1e-15) if f(1.0) > 0 else 1.0

print("library ready")


library ready


---
## Step 2 — Does the frontier stop moving?

Trace the achievable $(F_1,F_2)$ frontier at ancilla dimension 1, 2 and 3 by maximizing $wF_1+(1-w)F_2$. If
ancilla room is a resource extension, the frontier must complete and then stay put.


In [2]:
ws = [0.5, 0.65, 0.8, 0.95]
print("Achievable frontier vs ancilla dimension. Frontier = the observable for the set.\n")
print(f"{'w':>6} | " + " | ".join(f"{'da=' + str(d):>18}" for d in [1, 2, 3]))
print("-" * 70)
rows = {}
for w in ws:
    cells = []
    for da in [1, 2, 3]:
        (f1, f2), _ = optimize(lambda a, b: w * a + (1 - w) * b, da)
        rows[(w, da)] = (f1, f2); cells.append(f"{f1:.5f},{f2:.5f}")
    print(f"{w:>6} | " + " | ".join(f"{c:>18}" for c in cells))

d21 = max(max(abs(rows[(w,2)][0]-rows[(w,1)][0]), abs(rows[(w,2)][1]-rows[(w,1)][1])) for w in ws)
d32 = max(max(abs(rows[(w,3)][0]-rows[(w,2)][0]), abs(rows[(w,3)][1]-rows[(w,2)][1])) for w in ws)
print(f"\n   max frontier shift  da=1 -> da=2 :  {d21:.2e}")
print(f"   max frontier shift  da=2 -> da=3 :  {d32:.2e}   (optimizer noise)")
print("""
   The set grows once and then stops. Note the da=1 row at w=0.65 reaches a HIGHER F1
   (0.958 vs 0.948) but a much lower F2 -- it is a different point on a smaller
   frontier, and it loses on the weighted objective. The da=2 frontier dominates.
""")


Achievable frontier vs ancilla dimension. Frontier = the observable for the set.

     w |               da=1 |               da=2 |               da=3
----------------------------------------------------------------------
   0.5 |    0.81904,0.81903 |    0.83334,0.83333 |    0.83333,0.83334
  0.65 |    0.95802,0.63797 |    0.94766,0.68146 |    0.94766,0.68146
   0.8 |    0.99326,0.55071 |    0.99024,0.57422 |    0.99024,0.57424
  0.95 |    0.99976,0.50913 |    0.99963,0.51369 |    0.99964,0.51368

   max frontier shift  da=1 -> da=2 :  4.35e-02
   max frontier shift  da=2 -> da=3 :  1.80e-05   (optimizer noise)

   The set grows once and then stops. Note the da=1 row at w=0.65 reaches a HIGHER F1
   (0.958 vs 0.948) but a much lower F2 -- it is a different point on a smaller
   frontier, and it loses on the weighted objective. The da=2 frontier dominates.



---
## Step 3 — Does the Choi rank predict where it stopped?

The frontier completed at $d_a=2$. Nothing in that measurement knows about Stinespring. So compute the Choi
rank of the channel the optimizer actually found, at each ancilla dimension, and see whether the number matches.


In [3]:
print("Choi rank of the induced channel at each ancilla dimension.")
print("Stinespring: minimal environment dimension realizing a channel = rank(J).\n")
print(f"{'ancilla da':>11} {'F symmetric':>13} | {'largest Choi eigenvalues':>34} {'rank':>6}")
print("-" * 72)
for da in [1, 2, 3, 4]:
    (f1, f2), U = optimize(lambda a, b: 0.5 * (a + b), da, restarts=5)
    J = choi_from_unitary(U, da)
    ev = np.sort(np.linalg.eigvalsh(J))[::-1]
    rk = int((ev > 1e-7 * max(ev[0], 1e-12)).sum())
    print(f"{da:>11} {0.5*(f1+f2):>13.8f} | {' '.join(f'{e:6.3f}' for e in ev[:5]):>34} {rk:>6}")
print("""
   da=1 can only realize a rank-1 channel -- an isometry -- and tops out at 0.819.
   da=2 realizes rank 2, which IS the optimal channel's rank, and reaches 5/6.
   da=3 and da=4 still produce rank 2. The extra room goes unused.

   Saturation point = Choi rank. The criterion did not just label ancilla a resource;
   it said in advance where that resource would stop buying, and the number matched.
""")


Choi rank of the induced channel at each ancilla dimension.
Stinespring: minimal environment dimension realizing a channel = rank(J).

 ancilla da   F symmetric |           largest Choi eigenvalues   rank
------------------------------------------------------------------------
          1    0.81903559 |  2.000  0.000  0.000  0.000 -0.000      1
          2    0.83333333 |  1.000  1.000  0.000  0.000  0.000      2
          3    0.83333333 |  1.000  1.000  0.000  0.000  0.000      2
          4    0.83333332 |  1.000  1.000  0.000  0.000  0.000      2

   da=1 can only realize a rank-1 channel -- an isometry -- and tops out at 0.819.
   da=2 realizes rank 2, which IS the optimal channel's rank, and reaches 5/6.
   da=3 and da=4 still produce rank 2. The extra room goes unused.

   Saturation point = Choi rank. The criterion did not just label ancilla a resource;
   it said in advance where that resource would stop buying, and the number matched.



That is the part that makes this a criterion rather than a vocabulary. "Resource" now carries a prediction:
the achievable set completes at a computable dimension, and beyond it the coordinate is inert.

---
## Step 4 — Postselection moves the operation out of the set

Now the other kind. Build the exact probabilistic cloner explicitly — one Kraus operator from the dual basis —
and ask how large the success probability can be, using only the admissibility condition $A^\dagger A\le I$. The
Duan–Guo formula is used to check the answer, not to get it.


In [4]:
print("Exact cloner built explicitly; max p read off from A^dag A <= I.\n")
print(f"{'overlap x':>10} {'max p by construction':>22} {'1/(1+x)':>10} | {'clone error':>13}")
print("-" * 64)
ok = tot = 0
for x in [0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99]:
    p = max_p_by_construction(x)
    A, p1, p2 = cloner_kraus(x, p)
    err = max(np.linalg.norm(A @ p1 - np.sqrt(p) * np.kron(p1, p1)),
              np.linalg.norm(A @ p2 - np.sqrt(p) * np.kron(p2, p2)))
    good = abs(p - 1 / (1 + x)) < 1e-9; ok += good; tot += 1
    print(f"{x:>10} {p:>22.12f} {1/(1+x):>10.6f} | {err:>13.2e}  {'ok' if good else 'MISS'}")
print(f"\n   {ok} of {tot}. The clones are exact to 1e-16 and the ceiling on p lands on 1/(1+x).\n")

print("Which set does that operation belong to?")
print("Deterministic channels: Tr_out J = I exactly.  Heralded operations: Tr_out J <= I.\n")
print(f"{'overlap x':>10} {'p':>10} | {'eigenvalues of Tr_out J':>26} {'= I ?':>7} {'<= I ?':>7}")
print("-" * 68)
for x in [0.0, 0.3, 0.5, 0.9]:
    p = max_p_by_construction(x)
    A, _, _ = cloner_kraus(x, p)
    T = trace_out_output(choi_from_kraus(A)); ev = np.linalg.eigvalsh(T)
    print(f"{x:>10} {p:>10.6f} | {str(np.round(ev,6)):>26}"
          f" {str(np.allclose(T, np.eye(2), atol=1e-9)):>7} {str(bool((ev<=1+1e-9).all())):>7}")
print("""
   At x=0 it is trace preserving: a genuine deterministic channel (it is CNOT).
   At x>0 it is strictly trace non-increasing -- inside the larger set, outside the smaller.

   So postselection is not a point moving within an achievable set. It replaces the set
   being searched: trace-preserving -> trace-non-increasing. Class extension, measured.
""")


Exact cloner built explicitly; max p read off from A^dag A <= I.

 overlap x  max p by construction    1/(1+x) |   clone error
----------------------------------------------------------------
       0.0         1.000000000000   1.000000 |      0.00e+00  ok
       0.1         0.909090909091   0.909091 |      1.39e-17  ok
       0.3         0.769230769231   0.769231 |      2.22e-16  ok
       0.5         0.666666666667   0.666667 |      0.00e+00  ok
       0.7         0.588235294118   0.588235 |      1.20e-16  ok
       0.9         0.526315789474   0.526316 |      1.28e-16  ok
      0.99         0.502512562814   0.502513 |      2.56e-15  ok

   7 of 7. The clones are exact to 1e-16 and the ceiling on p lands on 1/(1+x).

Which set does that operation belong to?
Deterministic channels: Tr_out J = I exactly.  Heralded operations: Tr_out J <= I.

 overlap x          p |    eigenvalues of Tr_out J   = I ?  <= I ?
--------------------------------------------------------------------
       0.0

---
## Step 5 — Applying the criterion backwards, and correcting myself a third time

The criterion is now sharp enough to turn on the programme's own vocabulary, and it does not survive intact.

Across five notebooks I called $N$, $\tau$, $Q$, $G$, $M$ and noise "currencies" that "trade at exact rates."
By this criterion most of them are not currencies at all:

| What I called a currency | Does the transformation set grow? | Actually |
|---|---|---|
| Loop gain $G$ at fixed topology | no — same family $\{G\!\cdot\!H(s)\}$, different point | **resource coordinate** |
| Mass $M$ in a fixed network | no — same reaction network | **resource coordinate** |
| Damping $\zeta$ of an existing resonator | no — same 2-parameter family | **resource coordinate** |
| Adding a stage ($N\!\to\!N\!+\!1$) | yes — order-$N$ rational $\subsetneq$ order-$(N{+}1)$ | **class extension** |
| Adding delay ($\tau\ne0$) | yes — rational $\subsetneq$ rational $\times\,e^{-s\tau}$ | **class extension** |
| First-order $\to$ resonant crossing | yes — strictly more transfer functions | **class extension** |
| Ancilla dimension | no, past the Choi rank | resource coordinate that **saturates** |
| Postselection | yes — TP $\subsetneq$ TNI | **class extension** |

So the "exchange rate" $\arccos(1/G)/\sqrt{G^2-1}$ between stages and delay was never a trade inside one
market. It was a **map between two classes** that happen to hit the same phase target. Which is why the
language needed correcting three times: *bill* $\to$ *phase-closure requirement* $\to$ *gate and price* $\to$
this. Three revisions of one concept is not three discoveries; it is a measure of how loose the first version
was.

What survives, and is now precise, is a **two-level structure**:

$$\boxed{\;\text{A class extension opens a coordinate. The coordinate is what you then spend.}\;}$$

Admitting resonant crossings (class) opens $Q$ (coordinate). Admitting delay (class) opens $\tau$. Admitting
heralded operations (class) opens $p$. Admitting an environment (class) opens $d_a$ — which then saturates at
the Choi rank. Every number this programme has produced is a coordinate value, and every "currency" that
looked interchangeable was really an arrow between classes.

And that is the original claim, stated properly at last. The thread began with

```text
Sun()   ->   Sun(a, b, c, d)
```

**The arrow is a class extension. The values of $a,b,c,d$ are resources.** The programme spent five notebooks
conflating the two, which is exactly why "currency" kept needing repair: it was a word doing two jobs.

---
## Step 6 — What the criterion says about the earlier boundaries

With arrow and coordinate separated, the three categories sharpen.

| Boundary | Category | Reference class, named |
|---|---|---|
| $\sec(\pi/N)$, $2/Q$, $M_c$ | price | a coordinate in a fixed class; unbounded or bounded by the class |
| $e^\pi$ for diffusion | floor | coordinate exhausted *within* parabolic transport; the telegrapher **class extension** escapes it |
| $5/6$ for Copy | ceiling | coordinate exhausted within (exact, deterministic, universal, symmetric, $N{=}1$); each conjunct is a **class extension** away |
| $d_a$ past rank 2 | inert coordinate | the set is already complete; Stinespring says where |

Every one of them is a coordinate running out inside a class. The only difference between "floor" and
"ceiling" is whether the class extension that escapes it was easy to name — and both of the ones this
programme called absolute turned out to have one.

$$\boxed{\;\text{Nothing so far has been a boundary. Everything has been a coordinate exhausted inside a named class.}\;}$$

That is a stronger claim than the previous notebook's and a weaker one than the notebook before that, which is
the right direction of travel.

---
## Step 7 — Scoring

| Prediction | Result | Number |
|---|---|---|
| Achievable frontier stops growing | **confirmed** | shift $4.4\times10^{-2}$ then $2\times10^{-5}$ |
| Choi rank predicts the saturation point | **confirmed** | rank 2, saturation at $d_a=2$ |
| $d_a=1$ is rank-limited, not room-limited | **confirmed** | rank 1, tops out at 0.819 |
| Exact cloner exists by construction | **confirmed** | clones exact to $10^{-16}$ |
| Its max $p$ is $1/(1+x)$, from admissibility alone | **confirmed** | 7 of 7, 12 digits |
| Postselection leaves the CPTP set | **confirmed** | $\mathrm{Tr}_{\text{out}}J<I$ for $x>0$, $=I$ at $x=0$ |

**What is not new.** Stinespring dilation, Choi–Jamiołkowski, the rank-equals-minimal-environment fact, and
Duan–Guo are all standard. A quantum information theorist would read Steps 2–4 as exercises.

**What the exercise established.** That the resource/class distinction is *measurable*, and measuring it is
predictive rather than merely descriptive — the Choi rank told me where a resource would go inert before I
measured it. And then it turned on the programme's own vocabulary and removed most of it.

---
## Step 8 — Still open

- **Is there a boundary that is not a coordinate running out?** Six notebooks have not found one. The candidate
  test: an obstruction whose statement is invariant under *every* class extension that preserves some stated
  axiom, with the axiom named in advance rather than discovered afterwards. Linearity is the obvious candidate
  axiom for Copy, and naming it turns "ceiling" into "ceiling relative to linearity" — which is finally an
  honest label.
- **Global structure**, still, across seven notebooks. Every threshold is local; the coexisting latch-and-cycle
  from the second notebook remains the standing counterexample to treating local conditions as a full account.
  This is now the oldest unaddressed item in the programme and probably the one worth doing next.
- **Five reader failures, one pattern.** Mis-indexed steady state, biased sampler, principal-branch phase,
  one-sided fidelity, and a Monte Carlo missing a concentration peak. None found by reading code; all found by
  a number that could not be right. The rule: *when output is too good, too absolute, or violates a known limit,
  ask which quotient, branch, marginal or region the measurement discarded — before revising the model.*
